### Setup

In [1]:
import sagemaker
import boto3
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.s3 import S3Uploader

sagemaker_session = sagemaker.Session()
pipeline_session = PipelineSession()

region = boto3.Session().region_name
role = sagemaker.get_execution_role()

default_bucket = sagemaker_session.default_bucket()

bucket = sagemaker.Session().default_bucket()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
input_data_uri = S3Uploader.upload(
    local_path="../data/raw",
    desired_s3_uri=f"s3://{default_bucket}/pipeline-raw"
)

print(input_data_uri)

s3://sagemaker-us-east-1-110276528929/pipeline-raw


### Parámetros

In [3]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=1
)

instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.2xlarge"
)

model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval"
)

input_data = ParameterString(
    name="InputData",
    default_value=input_data_uri
)

batch_data = ParameterString(
    name="BatchData",
    default_value=f"s3://{default_bucket}/batch-data"
)

rmse_threshold = ParameterFloat(
    name="RmseThreshold",
    default_value=1.0
)

## ProcessingStep

### Processor

In [4]:
processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=1
)

In [5]:
from sagemaker.processing import ScriptProcessor

image_uri = "110276528929.dkr.ecr.us-east-1.amazonaws.com/ml-preprocessing-byoc:latest"

processor = ScriptProcessor(
    image_uri=image_uri,
    command=["python", "-m", "src.preprocessing"],
    instance_type="ml.m5.2xlarge",
    instance_count=processing_instance_count,
    base_job_name="prep-byoc",
    role=role,
    sagemaker_session=pipeline_session,
)

### Step args 

In [6]:
from sagemaker.processing import ProcessingInput, ProcessingOutput

processor = ScriptProcessor(
    image_uri=image_uri,
    command=["python", "-m", "src.preprocessing"],
    instance_type="ml.m5.2xlarge",
    instance_count=processing_instance_count,
    base_job_name="prep-byoc",
    role=role,
    sagemaker_session=pipeline_session,
)

processor_args = processor.run(
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/output/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test"
        ),
    ],
    code="../src/preprocessing/prep.py",
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


### Step

In [7]:
from sagemaker.workflow.steps import ProcessingStep

step_process = ProcessingStep(
    name="ProcessData",
    step_args=processor_args
)

## TrainingStep

### Estimator

In [8]:
from sagemaker.workflow.parameters import ParameterInteger

training_instance_count = ParameterInteger(
    name="TrainingInstanceCount",
    default_value=1
)

In [9]:
from sagemaker.estimator import Estimator

training_image_uri = "110276528929.dkr.ecr.us-east-1.amazonaws.com/ml-training-byoc:latest"

estimator = Estimator(
    image_uri=training_image_uri,
    role=role,
    instance_count=training_instance_count,
    instance_type="ml.m5.2xlarge",
    base_job_name="train-byoc",
    sagemaker_session=pipeline_session,
)

### Step args

In [10]:
train_args = estimator.fit(
    inputs={
        "train": step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
        "validation": step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
    }
)

### Step

In [11]:
from sagemaker.workflow.steps import TrainingStep

step_train = TrainingStep(
    name="TrainModel",
    step_args=train_args
)

## EvaluationStep

### Processor (evaluation)

In [12]:
evaluation_image_uri = image_uri

eval_processor = ScriptProcessor(
    image_uri=evaluation_image_uri,
    command=["python"],
    instance_type="ml.m5.2xlarge",
    instance_count=processing_instance_count,
    base_job_name="eval-byoc",
    role=role,
    sagemaker_session=pipeline_session,
)

### Step args

In [13]:
eval_args = eval_processor.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/input/model"
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/input/test"
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/output/evaluation"
        )
    ],
    code="../src/evaluation/evaluate.py",
)

### Step

In [14]:
from sagemaker.workflow.properties import PropertyFile

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

In [15]:
step_eval = ProcessingStep(
    name="EvaluateModel",
    step_args=eval_args,
    property_files=[evaluation_report],
)

## ModelStep

In [16]:
from sagemaker.model import Model

inference_image_uri = "110276528929.dkr.ecr.us-east-1.amazonaws.com/ml-inference-byoc:v2"

model = Model(
    image_uri=inference_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session,
)

### Step

In [17]:
from sagemaker.workflow.model_step import ModelStep

step_model = ModelStep(
    name="CreateModel",
    step_args=model.create()
)

## TransformStep

In [18]:
from sagemaker.transformer import Transformer
from sagemaker.workflow.steps import TransformStep
from sagemaker.inputs import TransformInput

transformer = Transformer(
    model_name=step_model.properties.ModelName,
    instance_type="ml.m5.large",
    instance_count=1,
    output_path=f"s3://{bucket}/transform-output",
    sagemaker_session=pipeline_session,
)

### Step

In [19]:
step_transform = TransformStep(
    name="BatchTransform",
    transformer=transformer,
    inputs=TransformInput(
        data=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
        content_type="text/csv"
    )
)

## ModelStep (registro) 

In [20]:
from sagemaker.model_metrics import ModelMetrics, MetricsSource

metrics_source = MetricsSource(
    s3_uri=step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
    content_type="application/json",
)

model_metrics = ModelMetrics(
    model_statistics=metrics_source
)

### Step

In [21]:
step_register = ModelStep(
    name="RegisterModel",
    step_args=model.register(
        content_types=["application/json"],
        response_types=["application/json"],
        inference_instances=["ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name="ml-model-group",
        model_metrics=model_metrics,
    )
)

## ConditionStep

In [22]:
from sagemaker.workflow.conditions import ConditionLessThan
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import JsonGet

### Threshold

In [23]:
rmse_threshold = 1.0

### Condición

In [24]:
condition = ConditionLessThan(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,  
        json_path="rmse"
    ),
    right=rmse_threshold
)

## FailStep

### Fail 

In [25]:
fail_step = FailStep(
    name="FailIfPoorModel",
    error_message="Modelo no cumple el threshold de RMSE"
)

### Step

In [26]:
step_condition = ConditionStep(
    name="CheckRMSE",
    conditions=[condition],
    if_steps=[step_register, step_transform],
    else_steps=[fail_step]
)

## Pruebas 

In [27]:
from sagemaker.processing import ScriptProcessor
import sagemaker

test_processor = ScriptProcessor(
    image_uri=image_uri,
    command=["python"],
    instance_type="ml.m5.2xlarge",
    instance_count=1,
    role=role,
    sagemaker_session=sagemaker.Session(),  
)

In [28]:
test_processor.run(
    code="../src/preprocessing/prep.py",
    inputs=[
        ProcessingInput(
            source=input_data_uri,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/output/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test"
        ),
    ],
)

INFO:sagemaker:Creating processing-job with name ml-preprocessing-byoc-2026-03-27-08-05-57-689


........2026-03-27 08:07:12,748 - __main__ - INFO - Iniciando pipeline de preparación (prep).
2026-03-27 08:07:12,748 - __main__ - INFO - Iniciando etapa: cargar_datos_raw
2026-03-27 08:07:14,191 - __main__ - INFO - train_raw: filas=2935849 cols=6
2026-03-27 08:07:14,209 - __main__ - INFO - test_raw: filas=214200 cols=3
2026-03-27 08:07:14,210 - __main__ - INFO - Etapa terminada: cargar_datos_raw duracion_seg=1.46
2026-03-27 08:07:14,210 - __main__ - INFO - Iniciando etapa: tipificar_y_filtrar
2026-03-27 08:07:14,759 - __main__ - INFO - train_clean: filas=2935845 cols=6
2026-03-27 08:07:14,775 - __main__ - INFO - test_clean: filas=214200 cols=3
2026-03-27 08:07:14,776 - __main__ - WARNING - Filas eliminadas por limpieza: train=4 test=0
2026-03-27 08:07:14,776 - __main__ - INFO - Etapa terminada: tipificar_y_filtrar duracion_seg=0.57
2026-03-27 08:07:14,776 - __main__ - INFO - Iniciando etapa: construir_panel_con_features
2026-03-27 08:07:28,265 - __main__ - INFO - panel: filas=7497000 

In [29]:
import sagemaker
from sagemaker.estimator import Estimator

# sesión normal para prueba aislada
test_session = sagemaker.Session()

# sacar los outputs del processing job que ya terminó bien
prep_desc = test_processor.latest_job.describe()
prep_outputs = {
    o["OutputName"]: o["S3Output"]["S3Uri"]
    for o in prep_desc["ProcessingOutputConfig"]["Outputs"]
}

print("TRAIN:", prep_outputs["train"])
print("VALIDATION:", prep_outputs["validation"])
print("TEST:", prep_outputs["test"])

# estimator de prueba
test_estimator = Estimator(
    image_uri=training_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.2xlarge",
    base_job_name="train-byoc-test",
    sagemaker_session=test_session,
)

# correr entrenamiento aislado
test_estimator.fit(
    inputs={
        "train": prep_outputs["train"],
        "validation": prep_outputs["validation"],
    },
    wait=True,
    logs=True,
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


TRAIN: s3://sagemaker-us-east-1-110276528929/ml-preprocessing-byoc-2026-03-27-08-05-57-689/output/train
VALIDATION: s3://sagemaker-us-east-1-110276528929/ml-preprocessing-byoc-2026-03-27-08-05-57-689/output/validation
TEST: s3://sagemaker-us-east-1-110276528929/ml-preprocessing-byoc-2026-03-27-08-05-57-689/output/test


INFO:sagemaker:Creating training-job with name: train-byoc-test-2026-03-27-08-08-50-293


2026-03-27 08:08:50 Starting - Starting the training job...
2026-03-27 08:09:13 Starting - Preparing the instances for training...
2026-03-27 08:09:50 Downloading - Downloading the training image
2026-03-27 08:09:50 Training - Training image download completed. Training in progress..Downloading ruff (10.6MiB)
 Downloaded ruff
Installed 8 packages in 7ms
2026-03-27 08:10:01,150 - __main__ - INFO - Iniciando entrenamiento.
2026-03-27 08:10:01,863 - __main__ - INFO - Datasets cargados. train_rows=7068600 valid_rows=214200
2026-03-27 08:10:01,866 - __main__ - INFO - Features=47 | cat_features=4
2026-03-27 08:10:01,867 - __main__ - INFO - Entrenando clasificador (venta vs no venta).
2026-03-27 08:12:08,441 - __main__ - INFO - Clasificador entrenado. prob_valid_mean=0.1109
2026-03-27 08:12:08,442 - __main__ - INFO - Entrenando regresor (unidades | venta).
2026-03-27 08:12:26,745 - __main__ - INFO - Regresor entrenado. mu_valid_mean=1.4657
2026-03-27 08:12:26,747 - __main__ - INFO - Alpha fij

In [30]:
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

test_session = sagemaker.Session()

train_model_s3 = test_estimator.model_data
test_s3 = prep_outputs["validation"]

print("MODEL:", train_model_s3)
print("TEST:", test_s3)

test_eval_processor = ScriptProcessor(
    image_uri=evaluation_image_uri,
    command=["python"],
    instance_type="ml.m5.2xlarge",
    instance_count=1,
    role=role,
    sagemaker_session=test_session,
)

test_eval_processor.run(
    code="../src/evaluation/evaluate.py",
    inputs=[
        ProcessingInput(
            source=train_model_s3,
            destination="/opt/ml/processing/input/model"
        ),
        ProcessingInput(
            source=test_s3,
            destination="/opt/ml/processing/input/test"
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/output/evaluation"
        )
    ],
    wait=True,
    logs=True,
)

MODEL: s3://sagemaker-us-east-1-110276528929/train-byoc-test-2026-03-27-08-08-50-293/output/model.tar.gz
TEST: s3://sagemaker-us-east-1-110276528929/ml-preprocessing-byoc-2026-03-27-08-05-57-689/output/validation


INFO:sagemaker:Creating processing-job with name ml-preprocessing-byoc-2026-03-27-08-13-09-973


......../opt/ml/processing/input/code/evaluate.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)



In [34]:
import sagemaker
from sagemaker.model import Model
from sagemaker.transformer import Transformer

test_session = sagemaker.Session()

test_model = Model(
    image_uri=inference_image_uri,
    model_data=test_estimator.model_data,
    role=role,
    sagemaker_session=test_session,
)

created_model = test_model.create()
print(created_model)

test_transformer = Transformer(
    model_name=test_model.name,
    instance_type="ml.m5.large",
    instance_count=1,
    output_path=f"s3://{bucket}/transform-output-test",
    sagemaker_session=test_session,
)

test_transformer.transform(
    data=prep_outputs["test"],
    content_type="text/csv",
    split_type="Line",
    wait=True,
    logs=True,
)

INFO:sagemaker:Creating model with name: ml-inference-byoc-2026-03-27-17-32-19-119
INFO:sagemaker:Creating transform job with name: ml-inference-byoc-2026-03-27-17-32-20-109


None
...........................[2026-03-27 17:36:44 +0000] [1] [INFO] Starting gunicorn 25.3.0
[2026-03-27 17:36:44 +0000] [1] [INFO] Listening at: http://0.0.0.0:8080 (1)
[2026-03-27 17:36:44 +0000] [1] [INFO] Using worker: sync
[2026-03-27 17:36:44 +0000] [7] [INFO] Booting worker with pid: 7
[2026-03-27 17:36:44 +0000] [1] [INFO] Control socket listening at /root/.gunicorn/gunicorn.ctl
2026-03-27 17:36:50,867 - src.inference.inference - INFO - Cargando modelo desde /opt/ml/model/model.joblib
2026-03-27 17:36:54,637 - src.inference.inference - INFO - Modelo cargado correctamente.
2026-03-27 17:36:54,637 - src.inference.inference - INFO - Health check OK.
2026-03-27 17:36:54,879 - src.inference.inference - INFO - Iniciando inferencia en tiempo real.
2026-03-27 17:36:55,086 - src.inference.inference - INFO - request_df: filas=27673 cols=47
2026-03-27 17:36:55,675 - src.inference.inference - INFO - Preds: n=27673 min=0.0224 p50=0.0570 max=15.5103
2026-03-27 17:36:55,707 - src.inference

## Pipeline

In [ ]:
from sagemaker.workflow.pipeline import Pipeline

pipeline = Pipeline(
    name="PipelineBYOC",
    parameters=[
        processing_instance_count,
        training_instance_count,
        input_data
    ],
    steps=[
        step_process,
        step_train,
        step_eval,
        step_model,
        step_condition
    ],
    sagemaker_session=pipeline_session,
)

## Ejecución

In [ ]:
pipeline.upsert(role_arn=role)

execution = pipeline.start()

execution.describe()

## Verificación

In [ ]:
execution.wait()

In [ ]:
execution.list_steps()